Generally we use ingestion system like:
1. (fixed schema + cloudFiles.schemaEvolutionMode = rescue) + (_rescued_data + _corrupt_record) == ERP, SAS

- Why?
    Because:
    - source contracts usually exist
    - structure mostly stable
    - but unexpected fields still happen
    - pipelines should remain resilient

- Typical systems
        ✔ ERP
        ✔ Salesforce
        ✔ SAP
        ✔ SaaS APIs
        ✔ CDC feeds
2. (strict schema + cloudFiles.schemaEvolutionMode = failOnNewColumns) + (_rescued_data + _corrupt_record) == Banking, follow the compliance.

- Why?
    Because:
    - unexpected data may indicate risk
    - audit/compliance requirements
    - downstream calculations sensitive

- Typical systems
        ✔ Banking core system
        ✔ Payment System
        ✔ Trading Capital market
        ✔ Insurance systems
        ✔ Healthcare systems
        ✔ Govermant/Tax tax

3. (no schema + cloudFiles.schemaEvolutionMode = addNewColumns + schemaHints) + (_rescued_data + _corrupt_record) == logs , event streams, telemetry, etc

- Why?
    Because:
    - no predefined schema
    - dynamic JSON payloads
    - frequent evolution

- Typical systems
        ✔ IoT
        ✔ clickstream
        ✔ application logs
        ✔ Kafka events
        ✔ telemetry

In [0]:
_schema = "dbfs:/Volumes/workspace/retails/raw_data/_schemas/retails/dev/order_items/"
_checkpoints = "dbfs:/Volumes/workspace/retails/raw_data/_checkpoints/retails/dev/bronze/order_items/"

In [0]:
schema = """
        order_item_id STRING,
        order_item_order_id STRING,
        order_item_product_id STRING,
        order_item_quantity STRING,
        order_item_subtotal STRING,
        order_item_product_price STRING
        """

# ingest order items by auto loader
df = (spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "json") 
    .option("cloudFiles.schemaLocation", _schema) \
    .option("cloudFiles.maxFilesPerTrigger", "1") \
    .schema(schema) \
    .option("cloudFiles.schemaEvolutionMode", "rescue") \
    .option("cloudFiles.rescuedDataColumn", "_rescued_data") \
    .option("cloudFiles.schemaHints", "order_item_id INTEGER, order_item_order_id INTEGER, order_item_product_id INTEGER, order_item_quantity INTEGER, order_item_subtotal FLOAT, order_item_product_price FLOAT, _corrupt_record STRING") \
    .load("dbfs:/Volumes/workspace/retails/raw_data/json/order_items/")
)


In [0]:
from pyspark.sql.functions import to_json, struct, col, current_timestamp, current_date, regexp_extract, expr, lit, StringType

df = df.withColumn("raw_data", to_json(struct("*"))) \
        .withColumn(
            "source_file_name",
            regexp_extract(col("_metadata.file_path"), r'([^/]+$)', 1)
        ) \
        .withColumn("source_file_path", col("_metadata.file_path")) \
        .withColumn("replay_flag", lit(False))


if "op" in df.columns:
    df = df.withColumn("op", col("op").cast(StringType()))
else:
    df = df.withColumn("op", lit("INSERT").cast(StringType()))


In [0]:
from pyspark.sql.functions import *
from datetime import datetime

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

def process_batch(df, batch_id):

    df = (
        df
        .withColumn("order_item_id", col("order_item_id").cast("String")) \
        .withColumn("order_item_order_id", col("order_item_order_id").cast("String")) \
        .withColumn("order_item_product_id", col("order_item_product_id").cast("String")) \
        .withColumn("order_item_quantity", col("order_item_quantity").cast("String")) \
        .withColumn("order_item_subtotal", col("order_item_subtotal").cast("String")) \
        .withColumn("order_item_product_price", col("order_item_product_price").cast("String")) \

        .withColumn("ingestion_ts", current_timestamp()) \
        .withColumn("ingestion_dt", current_date()) \
        .withColumn("batch_id", lit(batch_id)) \
        .withColumn("run_id", lit(run_id)) \
        .withColumn("source_system", lit("ERP"))
    )

    (
        df.write
            .format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable("retails.bronze.order_items_raw")
    )

In [0]:

(df.writeStream \
    .foreachBatch(process_batch) \
    .option("checkpointLocation", _checkpoints) \
    .trigger(availableNow=True)
    .start()
)

In [0]:
# GRANTS = "ALTER TABLE retails.bronze.order_items_raw OWNER TO `dp-sales-engineers`"
# spark.sql(GRANTS).display()

In [0]:
# dbutils.fs.ls(_schema)
# dbutils.fs.ls(_checkpoints)
# dbutils.fs.rm(_schema, True)
# dbutils.fs.rm(_checkpoints, True)

In [0]:
%sql
-- describe table extended retails.bronze.order_items_raw;
-- select count(*) from retails.bronze.order_items_raw;
-- select order_item_product_id, count(*) from retails.bronze.order_items_raw group by order_item_product_id;


In [0]:
# %sql
# CREATE OR REPLACE TABLE retails.bronze.order_items_raw (
#   order_item_id STRING,
#   order_item_order_id STRING,
#   order_item_product_id STRING,
#   order_item_quantity STRING,
#   order_item_subtotal STRING,
#   order_item_product_price STRING,
#   _corrupt_record string,
#   _rescued_data string,
#   raw_data string,
#   ingestion_ts timestamp,
#   ingestion_dt date,
#   batch_id int,
#   run_id string,
#   source_file_name string,
#   source_system string
# )
# USING DELTA
# PARTITIONED BY (ingestion_dt)
# LOCATION 's3://thoughtbulls-dp-uc-root-ap-south-1-8affd0fb/external_data/bronze/retails/order_items_raw'